In [ ]:
from project_utils import load_table, input_path, output_path, calculateCI, mean_error, standard_error, percentile_column, positive_fraction, SEED
import numpy as np
np.random.seed(SEED)


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm


In [ ]:
df = load_table('papers')


In [ ]:
df[df['if Prize']>0]


In [ ]:
df['Field'] = df['Field'].apply(lambda x:int(x//10))


In [ ]:
def ref_classify(x):
    if x <= 10:
        return  '1-10'
    elif x <= 20:
        return  '11-20'
    elif x <= 30:
        return  '21-30'
    else:
        return  '>30'
df['binref'] = df['outdegree'].apply(ref_classify)


In [ ]:
def ref_classify2(x):
    if x >= 1980 and x<1990:
        return  '1980-1989'
    elif x >= 1990 and x<2000:
        return  '1990-1999'
    elif x >= 2000 and x<2010:
        return  '2000-2009'
    else:
        return  '<1980'
df['binyear'] = df['year'].apply(ref_classify2)


In [ ]:
df.dropna(subset = 'RD' , inplace=True)


# issue 1, 2 References


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
sns.boxplot(x="binref", y="RD",order=['1-10','11-20','21-30','>30'],palette = ['#CD6155','#5499C7','#45B39D','#E59866'],
            data=df,meanline=True,linewidth = 2,fliersize=0.005)
sns.set_theme(style="white", palette=None)
plt.axhline(y = 0 , c = '#1A5276', linewidth = 1)
plt.ylim(0,10)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('Reference Number',fontsize = 30)
plt.ylabel('RDI',fontsize = 30)
plt.show()


In [ ]:
lst = [df[df['binref']==i]['RD'].values for i in df['binref'].unique()]
scipy.stats.f_oneway(lst[0],lst[1],lst[2],lst[3])


In [ ]:
formula = 'outdegree ~  RD'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df).fit()
params = model.params.values[1].round(3)
pvalues = model.pvalues.values[1].round(3)
print(params , pvalues)


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['RD'], df['outdegree'], c = 'black' , alpha = 0.01)
sns.regplot(data = df, y='outdegree' , x='RD',x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#1A5276',scatter = False)
plt.ylim(0,40)
plt.xlim(0,40)
plt.yticks([0,10,20,30,40],fontsize=20,)
plt.xticks([0,10,20,30,40],fontsize=20,)
plt.xlabel('RDI',fontsize = 30)
plt.ylabel('Reference Number',fontsize = 30)
plt.show()


# issue 34 Consistency in year and fields


In [ ]:
label = range(10)
dflst = [df[df['Field']==i] for i in label]


In [ ]:
scipy.stats.f_oneway(dflst[0]['RD'].values,dflst[1]['RD'].values,dflst[2]['RD'].values,dflst[3]['RD'].values,dflst[4]['RD'].values,dflst[5]['RD'].values,dflst[6]['RD'].values,dflst[7]['RD'].values,dflst[8]['RD'].values,dflst[9]['RD'].values,)


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
for i in label:
    sns.kdeplot(data=dflst[i], x="RD",label = label[i])
plt.ylim(0,1)
plt.xlim(0,10)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('RDI',fontsize = 30)
plt.ylabel('Density',fontsize = 30)
plt.legend(fontsize = 20)
plt.show()


In [ ]:
yearlabel = ['1980-1989','1990-1999','2000-2009']
dflst = [df[df['binyear']==i] for i in yearlabel]


In [ ]:
scipy.stats.f_oneway(dflst[0]['RD'].values,dflst[1]['RD'].values,dflst[2]['RD'].values,)


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
for i in range(3):
    sns.kdeplot(data=dflst[i], x="RD",label = yearlabel[i])
plt.ylim(0,1)
plt.xlim(0,10)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('RDI',fontsize = 30)
plt.ylabel('Density',fontsize = 30)
plt.legend(fontsize = 20)
plt.show()


# issue 5 Consistency citation count/ PageRank/ h


In [ ]:
topv = np.percentile(df['indegree'],99)
formula = 'indegree ~ RD'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df[df['indegree']<=topv]).fit()
print(model.summary())


In [ ]:
topv = np.percentile(df['indegree'],99)
formula = 'indegree ~  RD'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df[df['indegree']>topv]).fit()
print(model.summary())


In [ ]:
topv = np.percentile(df['indegree'],99)

f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['RD'], df['indegree'], c = 'black' , alpha = 0.01)
sns.regplot(data = df[df['indegree']>topv] , y='indegree' , x='RD', x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#922B21',scatter = False , label = 'Top 1% papers')
sns.regplot(data = df[df['indegree']<=topv] , y='indegree' , x='RD',x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#0E6655',scatter = False , label='Other papers')
plt.axhline(y = topv , c = '#1A5276', linestyle = '--', linewidth = 3)
# ax.set_yscale("log")
plt.ylim(0,250)
plt.xlim(0,20)
plt.yticks([0,50,100,150,200,250],fontsize=20,)
plt.xticks([0,5,10,15,20],fontsize=20,)
plt.xlabel('RDI',fontsize = 30)
plt.ylabel('Citation Count',fontsize = 30)
plt.legend(fontsize = 23)
plt.show()


In [ ]:
topv = np.percentile(df['h'],99)
formula = 'h ~ RD'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df[df['h']<=topv]).fit()
print(model.summary())


In [ ]:
topv = np.percentile(df['h'],99)
formula = 'h ~  RD'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df[df['h']>topv]).fit()
print(model.summary())


In [ ]:
topv = np.percentile(df['h'],99)

f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['RD'], df['h'], c = 'black' , alpha = 0.01)
sns.regplot(data = df[df['h']>topv] , y='h' , x='RD', x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#922B21',scatter = False, label = 'Top 1% papers')
sns.regplot(data = df[df['h']<=topv] , y='h' , x='RD',x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#0E6655',scatter = False, label='Other papers')
plt.axhline(y = topv , c = '#1A5276', linestyle = '--', linewidth = 3)
plt.ylim(0,40)
plt.xlim(0,20)
plt.yticks([0,10,20,30,40],fontsize=20,)
plt.xticks([0,5,10,15,20],fontsize=20,)
plt.xlabel('RDI',fontsize = 30)
plt.ylabel('h-index',fontsize = 30)
plt.legend(fontsize = 23)
plt.show()


In [ ]:
topv = np.percentile(df['pagerank'],99)
formula = 'pagerank ~ RD'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df[df['pagerank']<=topv]).fit()
print(model.summary())


In [ ]:
topv = np.percentile(df['pagerank'],99)
formula = 'pagerank ~  RD'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df[df['pagerank']>topv]).fit()
print(model.summary())


In [ ]:
topv = np.percentile(df['pagerank'],99)

f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['RD'], df['pagerank'], c = 'black' , alpha = 0.01)
sns.regplot(data = df[df['pagerank']>topv] , y='pagerank' , x='RD', x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#922B21',scatter = False, label = 'Top 1% papers')
sns.regplot(data = df[df['pagerank']<=topv] , y='pagerank' , x='RD',x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#0E6655',scatter = False, label='Other papers')
plt.axhline(y = topv , c = '#1A5276', linestyle = '--', linewidth = 3)
plt.ylim(0,0.00004)
plt.xlim(0,20)
plt.yticks([0,0.00001,0.00002,0.00003,0.00004],fontsize=20,)
plt.xticks([0,5,10,15,20],fontsize=20,)
plt.xlabel('RDI',fontsize = 30)
plt.ylabel('PageRank',fontsize = 30)
plt.legend(fontsize = 23)
plt.show()


# issue 6 Convergent validity milestones


In [ ]:
Prizedf = df[df['if Prize']==1]

f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['RD'], df['indegree'], c = 'black' , alpha = 0.01)
plt.scatter(df['RD'].values[100], df['indegree'].values[100], c = 'black' , alpha = 0.2,label = 'Other papers')
plt.scatter(Prizedf['RD'], Prizedf['indegree'], c = '#922B21' ,marker = '*',s=180, alpha = 1,label = 'Milestones')
ax.set_yscale("log")
plt.ylim(3,10000)
plt.xlim(0,15)
plt.yticks([10e0,10e1,10e2,10e3],fontsize=20,)
plt.xticks([0,5,10,15],fontsize=20,)
plt.xlabel('RDI',fontsize = 30)
plt.ylabel('Citation Count',fontsize = 30)
plt.legend(fontsize = 23)
plt.show()


In [ ]:
Prizedf = df[df['if Prize']==1]
f, ax = plt.subplots(figsize=(7, 7))
sns.histplot(data = Prizedf, x='RD',binwidth=.3,fill = True,color ='#922B21',label='Milestones' )
plt.ylim(0,6)
plt.xlim(0,15)
plt.yticks([0,2,4,6],fontsize=20,)
plt.xticks([0,5,10,15],fontsize=20,)
plt.xlabel('RDI',fontsize = 30)
plt.ylabel('Distribution',fontsize = 30)
plt.legend(fontsize = 23)
plt.show()


# issue 7 Citation Database


In [ ]:
mag = load_table('mag_papers')
if mag['id'].duplicated().any():
    raise ValueError('MAG paper ids must be unique')
df['Disruption'] = df['id'].map(mag.set_index('id')['Disruption'])
f, ax = plt.subplots(figsize=(7, 7))
sns.histplot(data = df, x='RD',binwidth=.01,fill = False,label='RDI in APS')
sns.histplot(data = df, x='Disruption',binwidth=.01,fill = False, label='DI in MAG')
plt.ylim(0,1000)
plt.yticks([0,200,400,600,800,1000],fontsize=20,)
plt.xticks([-1,-0.5,0,0.5,1],fontsize=20,)
plt.xlabel('Metric value',fontsize = 30)
plt.ylabel('Distribution',fontsize = 30)
plt.legend(fontsize = 23)
plt.show()
